# Task 3 - Part 2: Text Representation

This notebook generates three text representations from labeled sentiment data:
1. Bag-of-Words (BoW) with unigrams + bigrams
2. TF-IDF feature vectors with unigrams + bigrams
3. TF-IDF weighted GloVe document embeddings

Outputs are stored in both CSV and JSON formats.

## 1. Imports

In [19]:
from pathlib import Path
import json
import subprocess
import sys
from datetime import datetime, timezone

import gensim.downloader as api
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 2. Configuration

In [20]:
USE_ALL_RECORDS = True
N_RECORDS = 200
RANDOM_STATE = 42

BOW_NGRAM_RANGE = (1, 2)
TFIDF_NGRAM_RANGE = (1, 2)
GLOVE_MODEL_NAME = "glove-wiki-gigaword-100"
OUTPUT_DIR_NAME = "text_representation_outputs"

TASK3_DIR = Path(".")
PREPROCESSING_TEMP_DIR = TASK3_DIR / "preprocessing_temp"
PREPROCESSING_TEMP_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV_PATH = TASK3_DIR / "Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv"
WORKING_INPUT_CSV = PREPROCESSING_TEMP_DIR / "working_input_for_styles.csv"

STYLE_B_SCRIPT_PATH = TASK3_DIR / "preprocess_pipeline_B.py"
STYLE_C_SCRIPT_PATH = TASK3_DIR / "preprocess_pipeline_C.py"

ORIGINAL_STYLE_CSV = PREPROCESSING_TEMP_DIR / "Cleaned_Iran_War_Sentiment_style_original.csv"
STYLE_B_CSV = PREPROCESSING_TEMP_DIR / "Cleaned_Iran_War_Sentiment_style_b.csv"
STYLE_C_CSV = PREPROCESSING_TEMP_DIR / "Cleaned_Iran_War_Sentiment_style_c.csv"

OUTPUT_DIR = TASK3_DIR / OUTPUT_DIR_NAME
FEATURES_DIR = OUTPUT_DIR / "three_style_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_JSON = OUTPUT_DIR / "dataset_manifest.json"
STYLE_SUMMARY_JSON = FEATURES_DIR / "style_feature_summary.json"

print(f"Input CSV: {INPUT_CSV_PATH}")
print(f"Preprocessing temp dir: {PREPROCESSING_TEMP_DIR}")
print(f"Features output dir: {FEATURES_DIR}")
print(f"GloVe model: {GLOVE_MODEL_NAME}")

Input CSV: Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv
Preprocessing temp dir: preprocessing_temp
Features output dir: text_representation_outputs\three_style_features
GloVe model: glove-wiki-gigaword-100


## 3. Load and Validate Data

In [21]:
if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_CSV_PATH}")

df_raw = pd.read_csv(INPUT_CSV_PATH)
required_columns = ["final_text", "ground_truth"]
missing_columns = [col for col in required_columns if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_clean = df_raw.copy()
df_clean["final_text"] = df_clean["final_text"].fillna("").astype(str).str.strip()
df_clean["ground_truth"] = df_clean["ground_truth"].fillna("unknown").astype(str).str.strip()
df_clean = df_clean[df_clean["final_text"] != ""].copy()

if USE_ALL_RECORDS:
    df_work = df_clean.reset_index(drop=True)
else:
    n = min(N_RECORDS, len(df_clean))
    df_work = df_clean.sample(n=n, random_state=RANDOM_STATE).sort_index().reset_index(drop=True)

df_work.insert(0, "row_id", np.arange(len(df_work), dtype=int))

df_original_style = df_work.copy()
df_original_style["final_text_original"] = df_original_style["final_text"]
df_original_style.to_csv(ORIGINAL_STYLE_CSV, index=False)

print(f"Rows loaded: {len(df_raw)}")
print(f"Rows after cleaning/filtering: {len(df_work)}")
print("Label distribution:")
print(df_work["ground_truth"].value_counts())
print(f"Original style dataset saved to: {ORIGINAL_STYLE_CSV}")
df_work[["row_id", "final_text", "ground_truth"]].head()

Rows loaded: 500
Rows after cleaning/filtering: 491
Label distribution:
ground_truth
neutral     291
negative    195
positive      5
Name: count, dtype: int64
Original style dataset saved to: preprocessing_temp\Cleaned_Iran_War_Sentiment_style_original.csv


,row_id,final_text,ground_truth
0,0,belgium also say trumps war besides spain coun...,negative
1,1,massive war price tag could massive problem to...,negative
2,2,write american soldier kill innocent woman chi...,negative
3,3,verdant square radio playing note bowie hawkin...,neutral
4,4,one get away reckless stupid thing like attack...,negative


## 4. Save Dataset Manifest

In [22]:
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_csv": str(INPUT_CSV_PATH),
    "records_used": int(len(df_work)),
    "use_all_records": bool(USE_ALL_RECORDS),
    "bow_ngram_range": [BOW_NGRAM_RANGE[0], BOW_NGRAM_RANGE[1]],
    "tfidf_ngram_range": [TFIDF_NGRAM_RANGE[0], TFIDF_NGRAM_RANGE[1]],
    "style_files": {
        "original": str(ORIGINAL_STYLE_CSV),
        "style_b": str(STYLE_B_CSV),
        "style_c": str(STYLE_C_CSV)
    },
    "pipeline_scripts": {
        "style_b": str(STYLE_B_SCRIPT_PATH),
        "style_c": str(STYLE_C_SCRIPT_PATH)
    },
    "label_distribution": {k: int(v) for k, v in df_work["ground_truth"].value_counts().to_dict().items()}
}

with MANIFEST_JSON.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=True)

print(f"Manifest saved: {MANIFEST_JSON}")
manifest

Manifest saved: text_representation_outputs\dataset_manifest.json


{'created_utc': '2026-04-10T22:28:26.170436+00:00',
 'input_csv': 'Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv',
 'records_used': 491,
 'use_all_records': True,
 'bow_ngram_range': [1, 2],
 'tfidf_ngram_range': [1, 2],
 'style_files': {'original': 'preprocessing_temp\\Cleaned_Iran_War_Sentiment_style_original.csv',
  'style_b': 'preprocessing_temp\\Cleaned_Iran_War_Sentiment_style_b.csv',
  'style_c': 'preprocessing_temp\\Cleaned_Iran_War_Sentiment_style_c.csv'},
 'pipeline_scripts': {'style_b': 'preprocess_pipeline_B.py',
  'style_c': 'preprocess_pipeline_C.py'},
 'label_distribution': {'neutral': 291, 'negative': 195, 'positive': 5}}

## 5. Build 3 Preprocessed Datasets

In [23]:
if not STYLE_B_SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Missing script: {STYLE_B_SCRIPT_PATH}")
if not STYLE_C_SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Missing script: {STYLE_C_SCRIPT_PATH}")

df_work.to_csv(WORKING_INPUT_CSV, index=False)

style_b_cmd = [
    sys.executable,
    str(STYLE_B_SCRIPT_PATH),
    "--input",
    str(WORKING_INPUT_CSV),
    "--output",
    str(STYLE_B_CSV),
    "--text_column",
    "final_text",
    "--output_column",
    "final_text_style_b",
]

style_c_cmd = [
    sys.executable,
    str(STYLE_C_SCRIPT_PATH),
    "--input",
    str(WORKING_INPUT_CSV),
    "--output",
    str(STYLE_C_CSV),
    "--text_column",
    "final_text",
    "--output_column",
    "final_text_style_c",
    "--min_corpus_freq",
    "2",
]

style_b_run = subprocess.run(style_b_cmd, check=True, capture_output=True, text=True)
style_c_run = subprocess.run(style_c_cmd, check=True, capture_output=True, text=True)

print("Style B pipeline output:")
print(style_b_run.stdout.strip())
if style_b_run.stderr.strip():
    print("Style B warnings:")
    print(style_b_run.stderr.strip())

print("\nStyle C pipeline output:")
print(style_c_run.stdout.strip())
if style_c_run.stderr.strip():
    print("Style C warnings:")
    print(style_c_run.stderr.strip())

df_style_b = pd.read_csv(STYLE_B_CSV)
df_style_c = pd.read_csv(STYLE_C_CSV)

for required_col in ["row_id", "final_text_style_b"]:
    if required_col not in df_style_b.columns:
        raise ValueError(f"Missing column '{required_col}' in {STYLE_B_CSV}")

for required_col in ["row_id", "final_text_style_c"]:
    if required_col not in df_style_c.columns:
        raise ValueError(f"Missing column '{required_col}' in {STYLE_C_CSV}")

assert len(df_style_b) == len(df_work) == len(df_style_c), "Row count mismatch across preprocessed datasets."
assert df_style_b["row_id"].equals(df_work["row_id"]), "row_id mismatch for style_b dataset."
assert df_style_c["row_id"].equals(df_work["row_id"]), "row_id mismatch for style_c dataset."

style_texts = {
    "original": df_work["final_text"].fillna("").astype(str).str.strip(),
    "style_b": df_style_b["final_text_style_b"].fillna("").astype(str).str.strip(),
    "style_c": df_style_c["final_text_style_c"].fillna("").astype(str).str.strip(),
}

preprocessed_datasets = {
    "original": df_original_style,
    "style_b": df_style_b,
    "style_c": df_style_c,
}

base_text = df_work["final_text"].fillna("").astype(str).str.strip()
for style_name, series in style_texts.items():
    changed_rows = int((series != base_text).sum())
    print(f"{style_name}: {changed_rows} rows differ from original final_text")

style_preview = pd.DataFrame({
    "row_id": df_work["row_id"].head(5),
    "ground_truth": df_work["ground_truth"].head(5),
    "original": style_texts["original"].head(5).values,
    "style_b": style_texts["style_b"].head(5).values,
    "style_c": style_texts["style_c"].head(5).values,
})

style_preview

Style B pipeline output:
Pipeline B complete.
Input rows: 491
Output saved to: preprocessing_temp\Cleaned_Iran_War_Sentiment_style_b.csv
Output column: final_text_style_b
Empty processed rows: 0
Rows changed vs source column 'final_text': 425

Style C pipeline output:
Pipeline C complete.
Input rows: 491
Output saved to: preprocessing_temp\Cleaned_Iran_War_Sentiment_style_c.csv
Output column: final_text_style_c
Empty processed rows: 1
Rows changed vs source column 'final_text': 446
original: 0 rows differ from original final_text
style_b: 425 rows differ from original final_text
style_c: 446 rows differ from original final_text


,row_id,ground_truth,original,style_b,style_c
0,0,negative,belgium also say trumps war besides spain coun...,belgium also say trump war besid spain country...,belgium also say trumps war spain country soun...
1,1,negative,massive war price tag could massive problem to...,massive war price tag could massive problem to...,massive war price could problem top leader pro...
2,2,negative,write american soldier kill innocent woman chi...,write american soldier kill innocent woman chi...,write american soldier kill innocent woman chi...
3,3,neutral,verdant square radio playing note bowie hawkin...,verdant square radio play note bowie hawkin ep...,verdant square radio playing note bowie hawkin...
4,4,negative,one get away reckless stupid thing like attack...,one get away reckles stupid thing like attack ...,one get away stupid thing like attack iran wit...


## 6. BoW, TF-IDF, and GloVe for Each Style

In [28]:
style_feature_summary = []
bow_previews = {}
tfidf_previews = {}
glove_previews = {}

print(f"Loading GloVe model: {GLOVE_MODEL_NAME}")
glove_model = api.load(GLOVE_MODEL_NAME)
glove_dim = int(glove_model.vector_size)

for style_name, text_series in style_texts.items():
    text_series = text_series.fillna("").astype(str).str.strip()
    if int((text_series != "").sum()) == 0:
        raise ValueError(f"Style '{style_name}' has no non-empty text after preprocessing.")

    bow_vectorizer_style = CountVectorizer(ngram_range=BOW_NGRAM_RANGE)
    X_bow_style = bow_vectorizer_style.fit_transform(text_series)
    bow_columns = [f"bow_f{idx}" for idx in range(X_bow_style.shape[1])]

    bow_df_style = pd.DataFrame(X_bow_style.toarray().astype(np.int16), columns=bow_columns)
    bow_df_style.insert(0, "ground_truth", df_work["ground_truth"].values)
    bow_df_style.insert(0, "row_id", df_work["row_id"].values)

    bow_csv_path = FEATURES_DIR / f"bow_{style_name}.csv"
    bow_json_path = FEATURES_DIR / f"bow_{style_name}.json"
    bow_metadata_path = FEATURES_DIR / f"bow_{style_name}_metadata.json"
    bow_df_style.to_csv(bow_csv_path, index=False)
    bow_df_style.to_json(bow_json_path, orient="records", force_ascii=True)

    tfidf_vectorizer_style = TfidfVectorizer(ngram_range=TFIDF_NGRAM_RANGE)
    X_tfidf_style = tfidf_vectorizer_style.fit_transform(text_series)
    tfidf_columns = [f"tfidf_f{idx}" for idx in range(X_tfidf_style.shape[1])]

    tfidf_df_style = pd.DataFrame(X_tfidf_style.toarray().astype(np.float32), columns=tfidf_columns)
    tfidf_df_style.insert(0, "ground_truth", df_work["ground_truth"].values)
    tfidf_df_style.insert(0, "row_id", df_work["row_id"].values)

    tfidf_csv_path = FEATURES_DIR / f"tfidf_{style_name}.csv"
    tfidf_json_path = FEATURES_DIR / f"tfidf_{style_name}.json"
    tfidf_df_style.to_csv(tfidf_csv_path, index=False)
    tfidf_df_style.to_json(tfidf_json_path, orient="records", force_ascii=True)

    vocab = tfidf_vectorizer_style.vocabulary_
    idf = tfidf_vectorizer_style.idf_
    term_idf = {term: float(idf[idx]) for term, idx in vocab.items()}

    glove_vectors = np.zeros((len(text_series), glove_dim), dtype=np.float32)
    glove_tokens_covered = 0
    glove_tokens_total = 0

    for row_idx, text in enumerate(text_series):
        weighted_sum = np.zeros(glove_dim, dtype=np.float32)
        weight_total = 0.0
        for token in text.split():
            glove_tokens_total += 1
            if token in glove_model:
                weight = term_idf.get(token, 1.0)
                weighted_sum += glove_model[token] * weight
                weight_total += weight
                glove_tokens_covered += 1
        if weight_total > 0.0:
            glove_vectors[row_idx] = weighted_sum / weight_total

    glove_columns = [f"glove_f{idx}" for idx in range(glove_dim)]
    glove_df_style = pd.DataFrame(glove_vectors, columns=glove_columns)
    glove_df_style.insert(0, "ground_truth", df_work["ground_truth"].values)
    glove_df_style.insert(0, "row_id", df_work["row_id"].values)

    glove_csv_path = FEATURES_DIR / f"glove_{style_name}.csv"
    glove_json_path = FEATURES_DIR / f"glove_{style_name}.json"
    glove_metadata_path = FEATURES_DIR / f"glove_{style_name}_metadata.json"
    glove_df_style.to_csv(glove_csv_path, index=False)
    glove_df_style.to_json(glove_json_path, orient="records", force_ascii=True)

    bow_previews[style_name] = bow_df_style.iloc[:3, :12].copy()
    tfidf_previews[style_name] = tfidf_df_style.iloc[:3, :12].copy()
    glove_previews[style_name] = glove_df_style.iloc[:3, :12].copy()

    bow_density = float(X_bow_style.nnz / (X_bow_style.shape[0] * X_bow_style.shape[1])) if X_bow_style.shape[1] > 0 else 0.0
    tfidf_density = float(X_tfidf_style.nnz / (X_tfidf_style.shape[0] * X_tfidf_style.shape[1])) if X_tfidf_style.shape[1] > 0 else 0.0
    glove_non_zero = int(np.count_nonzero(glove_vectors))
    glove_density = float(glove_non_zero / (glove_vectors.shape[0] * glove_vectors.shape[1])) if glove_vectors.shape[1] > 0 else 0.0
    coverage_rate = float(glove_tokens_covered / glove_tokens_total) if glove_tokens_total > 0 else 0.0

    bow_metadata_payload = {
        "style": style_name,
        "representation": "bow",
        "ngram_range": [int(BOW_NGRAM_RANGE[0]), int(BOW_NGRAM_RANGE[1])],
        "rows": int(X_bow_style.shape[0]),
        "features": int(X_bow_style.shape[1]),
        "non_zero_entries": int(X_bow_style.nnz),
        "density": bow_density,
        "feature_name_sample": bow_vectorizer_style.get_feature_names_out()[:50].tolist(),
    }
    with bow_metadata_path.open("w", encoding="utf-8") as f:
        json.dump(bow_metadata_payload, f, indent=2, ensure_ascii=True)

    glove_metadata_payload = {
        "style": style_name,
        "representation": "glove",
        "model_name": GLOVE_MODEL_NAME,
        "rows": int(glove_vectors.shape[0]),
        "dimensions": int(glove_vectors.shape[1]),
        "non_zero_entries": glove_non_zero,
        "density": glove_density,
        "tokens_total": int(glove_tokens_total),
        "tokens_covered": int(glove_tokens_covered),
        "coverage_rate": coverage_rate,
    }
    with glove_metadata_path.open("w", encoding="utf-8") as f:
        json.dump(glove_metadata_payload, f, indent=2, ensure_ascii=True)

    style_feature_summary.append({
        "style": style_name,
        "rows": int(len(text_series)),
        "bow_features": int(X_bow_style.shape[1]),
        "bow_non_zero_entries": int(X_bow_style.nnz),
        "bow_density": bow_density,
        "bow_csv": str(bow_csv_path),
        "bow_json": str(bow_json_path),
        "bow_metadata_json": str(bow_metadata_path),
        "bow_metadata": bow_metadata_payload,
        "tfidf_features": int(X_tfidf_style.shape[1]),
        "tfidf_non_zero_entries": int(X_tfidf_style.nnz),
        "tfidf_density": tfidf_density,
        "tfidf_csv": str(tfidf_csv_path),
        "tfidf_json": str(tfidf_json_path),
        "glove_dimensions": glove_dim,
        "glove_non_zero_entries": glove_non_zero,
        "glove_density": glove_density,
        "glove_tokens_total": int(glove_tokens_total),
        "glove_tokens_covered": int(glove_tokens_covered),
        "glove_coverage_rate": coverage_rate,
        "glove_csv": str(glove_csv_path),
        "glove_json": str(glove_json_path),
        "glove_metadata_json": str(glove_metadata_path),
        "glove_metadata": glove_metadata_payload,
    })

style_feature_summary_df = pd.DataFrame(style_feature_summary)
print(f"Feature files saved in: {FEATURES_DIR}")
style_feature_summary_df

Loading GloVe model: glove-wiki-gigaword-100
Feature files saved in: text_representation_outputs\three_style_features


,style,rows,bow_features,bow_non_zero_entries,bow_density,bow_csv,bow_json,bow_metadata_json,bow_metadata,tfidf_features,...,glove_dimensions,glove_non_zero_entries,glove_density,glove_tokens_total,glove_tokens_covered,glove_coverage_rate,glove_csv,glove_json,glove_metadata_json,glove_metadata
0,original,491,11036,19295,0.003561,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'original', 'representation': 'bow',...",11036,...,100,49100,1.000000,10680,10647,0.996910,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'original', 'representation': 'glove..."
1,style_b,491,11287,20690,0.003733,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_b', 'representation': 'bow', ...",11287,...,100,49100,1.000000,11166,10252,0.918144,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_b', 'representation': 'glove'..."
2,style_c,491,8301,18066,0.004433,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_c', 'representation': 'bow', ...",8301,...,100,49000,0.997963,9844,8852,0.899228,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_c', 'representation': 'glove'..."


In [29]:
style_summary_payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_csv": str(INPUT_CSV_PATH),
    "records_used": int(len(df_work)),
    "representations": ["bow", "tfidf", "glove"],
    "styles": style_feature_summary,
}

with STYLE_SUMMARY_JSON.open("w", encoding="utf-8") as f:
    json.dump(style_summary_payload, f, indent=2, ensure_ascii=True)

print(f"Style feature summary saved: {STYLE_SUMMARY_JSON}")
style_summary_payload

Style feature summary saved: text_representation_outputs\three_style_features\style_feature_summary.json


{'created_utc': '2026-04-10T22:37:56.936943+00:00',
 'input_csv': 'Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv',
 'records_used': 491,
 'representations': ['bow', 'tfidf', 'glove'],
 'styles': [{'style': 'original',
   'rows': 491,
   'bow_features': 11036,
   'bow_non_zero_entries': 19295,
   'bow_density': 0.0035608329414786935,
   'bow_csv': 'text_representation_outputs\\three_style_features\\bow_original.csv',
   'bow_json': 'text_representation_outputs\\three_style_features\\bow_original.json',
   'bow_metadata_json': 'text_representation_outputs\\three_style_features\\bow_original_metadata.json',
   'bow_metadata': {'style': 'original',
    'representation': 'bow',
    'ngram_range': [1, 2],
    'rows': 491,
    'features': 11036,
    'non_zero_entries': 19295,
    'density': 0.0035608329414786935,
    'feature_name_sample': ['abandon',
     'abandon general',
     'abdul',
     'abdul mali',
     'able',
     'able contribute',
     'abroad',
     'abroad race',
     'a

## 7. Validation and Quick Preview

In [30]:
assert len(style_texts) == 3, "Expected 3 preprocessed text styles."
assert style_feature_summary_df["rows"].nunique() == 1, "Row count differs across styles."
assert int(style_feature_summary_df["rows"].iloc[0]) == len(df_work), "Feature rows do not match working dataset."
assert (style_feature_summary_df["bow_features"] > 0).all(), "At least one style has empty BoW vocabulary."
assert (style_feature_summary_df["tfidf_features"] > 0).all(), "At least one style has empty TF-IDF vocabulary."
assert (style_feature_summary_df["glove_dimensions"] > 0).all(), "At least one style has invalid GloVe dimensions."

for _, row in style_feature_summary_df.iterrows():
    required_files = [
        row["bow_csv"],
        row["bow_json"],
        row["bow_metadata_json"],
        row["tfidf_csv"],
        row["tfidf_json"],
        row["glove_csv"],
        row["glove_json"],
        row["glove_metadata_json"],
    ]
    for file_path in required_files:
        if not Path(file_path).exists():
            raise FileNotFoundError(f"Missing output file: {file_path}")

print("Validation passed for BoW, TF-IDF, and GloVe across all three preprocessing styles.")
style_feature_summary_df

Validation passed for BoW, TF-IDF, and GloVe across all three preprocessing styles.


,style,rows,bow_features,bow_non_zero_entries,bow_density,bow_csv,bow_json,bow_metadata_json,bow_metadata,tfidf_features,...,glove_dimensions,glove_non_zero_entries,glove_density,glove_tokens_total,glove_tokens_covered,glove_coverage_rate,glove_csv,glove_json,glove_metadata_json,glove_metadata
0,original,491,11036,19295,0.003561,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'original', 'representation': 'bow',...",11036,...,100,49100,1.000000,10680,10647,0.996910,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'original', 'representation': 'glove..."
1,style_b,491,11287,20690,0.003733,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_b', 'representation': 'bow', ...",11287,...,100,49100,1.000000,11166,10252,0.918144,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_b', 'representation': 'glove'..."
2,style_c,491,8301,18066,0.004433,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_c', 'representation': 'bow', ...",8301,...,100,49000,0.997963,9844,8852,0.899228,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,text_representation_outputs\three_style_featur...,"{'style': 'style_c', 'representation': 'glove'..."


In [27]:
print("Preprocessed text preview (first 5 rows):")
display(style_preview)

for style_name in ["original", "style_b", "style_c"]:
    print(f"\nBoW preview - {style_name}")
    display(bow_previews[style_name])
    print(f"TF-IDF preview - {style_name}")
    display(tfidf_previews[style_name])
    print(f"GloVe preview - {style_name}")
    display(glove_previews[style_name])

Preprocessed text preview (first 5 rows):


,row_id,ground_truth,original,style_b,style_c
0,0,negative,belgium also say trumps war besides spain coun...,belgium also say trump war besid spain country...,belgium also say trumps war spain country soun...
1,1,negative,massive war price tag could massive problem to...,massive war price tag could massive problem to...,massive war price could problem top leader pro...
2,2,negative,write american soldier kill innocent woman chi...,write american soldier kill innocent woman chi...,write american soldier kill innocent woman chi...
3,3,neutral,verdant square radio playing note bowie hawkin...,verdant square radio play note bowie hawkin ep...,verdant square radio playing note bowie hawkin...
4,4,negative,one get away reckless stupid thing like attack...,one get away reckles stupid thing like attack ...,one get away stupid thing like attack iran wit...



BoW preview - original


,row_id,ground_truth,bow_f0,bow_f1,bow_f2,bow_f3,bow_f4,bow_f5,bow_f6,bow_f7,bow_f8,bow_f9
0,0,negative,0,0,0,0,0,0,0,0,0,0
1,1,negative,0,0,0,0,0,0,0,0,0,0
2,2,negative,0,0,0,0,0,0,0,0,0,0


TF-IDF preview - original


...

,row_id,ground_truth,bow_f0,bow_f1,bow_f2,bow_f3,bow_f4,bow_f5,bow_f6,bow_f7,bow_f8,bow_f9
0,0,negative,0,0,0,0,0,0,0,0,0,0
1,1,negative,0,0,0,0,0,0,0,0,0,0
2,2,negative,0,0,0,0,0,0,0,0,0,0


TF-IDF preview - style_c


,row_id,ground_truth,tfidf_f0,tfidf_f1,tfidf_f2,tfidf_f3,tfidf_f4,tfidf_f5,tfidf_f6,tfidf_f7,tfidf_f8,tfidf_f9
0,0,negative,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,negative,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,negative,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


GloVe preview - style_c


,row_id,ground_truth,glove_f0,glove_f1,glove_f2,glove_f3,glove_f4,glove_f5,glove_f6,glove_f7,glove_f8,glove_f9
0,0,negative,-0.032382,0.274132,0.355910,-0.114193,0.020564,0.001246,-0.182478,0.050592,0.088692,0.112172
1,1,negative,-0.198964,0.193902,0.417208,-0.239247,0.176536,-0.219458,-0.232318,-0.270239,-0.110017,0.052301
2,2,negative,0.013643,0.362860,0.565773,-0.011474,-0.012272,0.046013,-0.201356,-0.185888,0.184837,0.218011
